# Snowpark Python Workshop: ShopStream Retail Pipeline

**Duration:** 30 minutes | **Level:** Intermediate

## Story
ShopStream is a growing e-commerce company. We'll build a complete Python-based data pipeline on Snowflake using Snowpark:

1. **DataFrames** - Explore and transform retail data
2. **UDFs** - Custom business logic (discount tier calculation)
3. **Stored Procedures** - Production-ready ETL pipeline

---

## Prerequisites
- Run `01_setup.sql` in Snowsight before starting this notebook
- Install: `pip install snowflake-snowpark-python`

## 1. Connect to Snowflake

In [1]:
from snowflake.snowpark import Session
from snowflake.snowpark.functions import (
    col, lit, sum, avg, count, count_distinct,
    date_trunc, when, upper, udf, pandas_udf, sproc
)
from snowflake.snowpark.types import (
    StringType, FloatType, PandasSeriesType
)

# Create session using connection name from ~/.snowflake/connections.toml
session = Session.builder.config("connection_name", "snowflake-enabled-trial").getOrCreate()

# Set session context
session.sql("USE WAREHOUSE SHOPSTREAM_WH").collect()
session.sql("USE DATABASE SHOPSTREAM").collect()
session.sql("USE SCHEMA RAW").collect()

print(f"Connected! Role: {session.get_current_role()}")
print(f"Warehouse: {session.get_current_warehouse()}")
print(f"Database: {session.get_current_database()}")

Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...
Connected! Role: "ACCOUNTADMIN"
Warehouse: "SHOPSTREAM_WH"
Database: "SHOPSTREAM"


## 2. Loading Data as DataFrames

Key insight: `session.table()` returns a **lazy** DataFrame. No query runs until you call `.show()`, `.collect()`, or `.count()`.

In [2]:
# Load tables - these are LAZY (no query runs yet!)
customers = session.table("SHOPSTREAM.RAW.CUSTOMERS")
products = session.table("SHOPSTREAM.RAW.PRODUCTS")
orders = session.table("SHOPSTREAM.RAW.ORDERS")

# .show() triggers execution
print(f"Customers: {customers.count()} rows")
print(f"Products: {products.count()} rows")
print(f"Orders: {orders.count()} rows")

customers.show(5)

Customers: 50 rows
Products: 20 rows
Orders: 200 rows
-----------------------------------------------------------------------------------------------
|"CUSTOMER_ID"  |"FIRST_NAME"  |"LAST_NAME"  |"EMAIL"              |"REGION"  |"SIGNUP_DATE"  |
-----------------------------------------------------------------------------------------------
|1              |Alice         |Johnson      |alice.j@email.com    |North     |2023-01-15     |
|2              |Bob           |Smith        |bob.s@email.com      |South     |2023-02-20     |
|3              |Charlie       |Brown        |charlie.b@email.com  |East      |2023-01-10     |
|4              |Diana         |Lee          |diana.l@email.com    |West      |2023-03-05     |
|5              |Eve           |Davis        |eve.d@email.com      |North     |2023-04-12     |
-----------------------------------------------------------------------------------------------



## 3. Select, Filter, and Transform

Use `col()` to reference columns. Chain operations like `.select()`, `.filter()`, `.with_column()`.

In [3]:
# Select specific columns
customers.select("FIRST_NAME", "REGION", "SIGNUP_DATE").show(5)

-------------------------------------------
|"FIRST_NAME"  |"REGION"  |"SIGNUP_DATE"  |
-------------------------------------------
|Alice         |North     |2023-01-15     |
|Bob           |South     |2023-02-20     |
|Charlie       |East      |2023-01-10     |
|Diana         |West      |2023-03-05     |
|Eve           |North     |2023-04-12     |
-------------------------------------------



In [4]:
# Filter: customers in the North region
north_customers = customers.filter(col("REGION") == "North")
print(f"North region customers: {north_customers.count()}")
north_customers.show(5)

North region customers: 13
---------------------------------------------------------------------------------------------
|"CUSTOMER_ID"  |"FIRST_NAME"  |"LAST_NAME"  |"EMAIL"            |"REGION"  |"SIGNUP_DATE"  |
---------------------------------------------------------------------------------------------
|1              |Alice         |Johnson      |alice.j@email.com  |North     |2023-01-15     |
|5              |Eve           |Davis        |eve.d@email.com    |North     |2023-04-12     |
|9              |Ivy           |Anderson     |ivy.a@email.com    |North     |2023-06-22     |
|13             |Mia           |Clark        |mia.c@email.com    |North     |2023-08-03     |
|17             |Quinn         |Young        |quinn.y@email.com  |North     |2023-06-10     |
---------------------------------------------------------------------------------------------



In [5]:
# Add computed columns
products_with_tax = products.with_column(
    "PRICE_WITH_TAX",
    col("UNIT_PRICE") * lit(1.08)
)
products_with_tax.select("PRODUCT_NAME", "UNIT_PRICE", "PRICE_WITH_TAX").show(5)

---------------------------------------------------------
|"PRODUCT_NAME"       |"UNIT_PRICE"  |"PRICE_WITH_TAX"  |
---------------------------------------------------------
|Wireless Mouse       |29.99         |32.3892           |
|USB-C Hub            |49.99         |53.9892           |
|Mechanical Keyboard  |89.99         |97.1892           |
|Monitor Stand        |39.99         |43.1892           |
|Webcam HD            |59.99         |64.7892           |
---------------------------------------------------------



In [6]:
# Conditional logic with WHEN
products_tiered = products.select(
    col("PRODUCT_NAME"),
    col("UNIT_PRICE"),
    when(col("UNIT_PRICE") > 200, "Premium")
    .when(col("UNIT_PRICE") > 50, "Standard")
    .otherwise("Budget")
    .alias("PRICE_TIER")
)
products_tiered.show(10)

---------------------------------------------------------
|"PRODUCT_NAME"           |"UNIT_PRICE"  |"PRICE_TIER"  |
---------------------------------------------------------
|Wireless Mouse           |29.99         |Budget        |
|USB-C Hub                |49.99         |Budget        |
|Mechanical Keyboard      |89.99         |Standard      |
|Monitor Stand            |39.99         |Budget        |
|Webcam HD                |59.99         |Standard      |
|Desk Lamp                |34.99         |Budget        |
|Ergonomic Chair          |299.99        |Premium       |
|Standing Desk            |449.99        |Premium       |
|Noise-Cancel Headphones  |199.99        |Standard      |
|Laptop Sleeve            |24.99         |Budget        |
---------------------------------------------------------



## 4. Joins

Join orders with customers and products to get a full picture.

In [7]:
# Build a denormalized order details view
order_details = (
    orders
    .join(customers, orders["CUSTOMER_ID"] == customers["CUSTOMER_ID"])
    .join(products, orders["PRODUCT_ID"] == products["PRODUCT_ID"])
    .select(
        orders["ORDER_ID"],
        customers["FIRST_NAME"],
        customers["REGION"],
        products["PRODUCT_NAME"],
        products["CATEGORY"],
        orders["QUANTITY"],
        (orders["QUANTITY"] * products["UNIT_PRICE"]).alias("ORDER_TOTAL"),
        orders["ORDER_DATE"]
    )
)

order_details.show(10)

----------------------------------------------------------------------------------------------------------------------------
|"ORDER_ID"  |"FIRST_NAME"  |"REGION"  |"PRODUCT_NAME"           |"CATEGORY"   |"QUANTITY"  |"ORDER_TOTAL"  |"ORDER_DATE"  |
----------------------------------------------------------------------------------------------------------------------------
|1001        |Alice         |North     |Wireless Mouse           |Electronics  |2           |59.98          |2024-01-05    |
|1002        |Bob           |South     |Mechanical Keyboard      |Electronics  |1           |89.99          |2024-01-08    |
|1003        |Charlie       |East      |Ergonomic Chair          |Furniture    |1           |299.99         |2024-01-10    |
|1004        |Diana         |West      |USB-C Hub                |Electronics  |3           |149.97         |2024-01-12    |
|1005        |Eve           |North     |Noise-Cancel Headphones  |Electronics  |1           |199.99         |2024-01-15    |


## 5. Aggregations

Group by region, month, product - and calculate metrics.

In [8]:
# Revenue by region
revenue_by_region = (
    order_details
    .group_by("REGION")
    .agg(
        sum("ORDER_TOTAL").alias("TOTAL_REVENUE"),
        count("ORDER_ID").alias("ORDER_COUNT"),
        avg("ORDER_TOTAL").alias("AVG_ORDER_VALUE")
    )
    .sort(col("TOTAL_REVENUE").desc())
)

revenue_by_region.show()

------------------------------------------------------------------
|"REGION"  |"TOTAL_REVENUE"  |"ORDER_COUNT"  |"AVG_ORDER_VALUE"  |
------------------------------------------------------------------
|South     |6701.24          |52             |128.87000000       |
|North     |6240.19          |52             |120.00365385       |
|West      |6160.23          |48             |128.33812500       |
|East      |5892.24          |48             |122.75500000       |
------------------------------------------------------------------



In [9]:
# Monthly revenue trend
monthly_revenue = (
    order_details
    .group_by(date_trunc("MONTH", col("ORDER_DATE")).alias("MONTH"))
    .agg(
        sum("ORDER_TOTAL").alias("REVENUE"),
        count_distinct("FIRST_NAME").alias("UNIQUE_CUSTOMERS")
    )
    .sort("MONTH")
)

monthly_revenue.show(12)

-----------------------------------------------
|"MONTH"     |"REVENUE"  |"UNIQUE_CUSTOMERS"  |
-----------------------------------------------
|2024-01-01  |2211.76    |12                  |
|2024-02-01  |2100.74    |14                  |
|2024-03-01  |2116.65    |16                  |
|2024-04-01  |2303.70    |17                  |
|2024-05-01  |2096.73    |16                  |
|2024-06-01  |1950.70    |17                  |
|2024-07-01  |2575.71    |16                  |
|2024-08-01  |2114.73    |17                  |
|2024-09-01  |2550.73    |18                  |
|2024-10-01  |1560.82    |12                  |
|2024-11-01  |1605.81    |13                  |
|2024-12-01  |1805.82    |12                  |
-----------------------------------------------



## 6. User-Defined Functions (UDFs)

Extend Snowflake with custom Python logic. UDFs run **on Snowflake's compute**, not locally.

### Scalar UDF: Discount Tier Calculator

In [10]:
# Define a scalar UDF
@udf(
    name="discount_tier",
    is_permanent=False,
    replace=True,
    input_types=[FloatType()],
    return_type=StringType()
)
def discount_tier(total_spend: float) -> str:
    """Assign discount tier based on total spend."""
    if total_spend >= 500:
        return "Platinum"
    elif total_spend >= 200:
        return "Gold"
    elif total_spend >= 100:
        return "Silver"
    else:
        return "Bronze"

print("UDF 'discount_tier' registered!")

UDF 'discount_tier' registered!


In [11]:
# Calculate total spend per customer
customer_spend = (
    orders
    .join(products, orders["PRODUCT_ID"] == products["PRODUCT_ID"])
    .group_by(orders["CUSTOMER_ID"])
    .agg(sum(col("QUANTITY") * col("UNIT_PRICE")).alias("TOTAL_SPEND"))
)

# Apply UDF
customer_tiers = (
    customer_spend
    .join(customers, customer_spend["CUSTOMER_ID"] == customers["CUSTOMER_ID"])
    .select(
        customers["FIRST_NAME"],
        customers["REGION"],
        customer_spend["TOTAL_SPEND"],
        discount_tier(col("TOTAL_SPEND")).alias("DISCOUNT_TIER")
    )
    .sort(col("TOTAL_SPEND").desc())
)

customer_tiers.show(15)

-------------------------------------------------------------
|"FIRST_NAME"  |"REGION"  |"TOTAL_SPEND"  |"DISCOUNT_TIER"  |
-------------------------------------------------------------
|Maya          |East      |1039.95        |Platinum         |
|Eve           |North     |1039.94        |Platinum         |
|Brian         |West      |1019.95        |Platinum         |
|Ursula        |East      |908.92         |Platinum         |
|Henry         |West      |798.92         |Platinum         |
|Rachel        |South     |744.96         |Platinum         |
|Olive         |North     |739.95         |Platinum         |
|Jack          |South     |710.95         |Platinum         |
|Hugo          |South     |689.94         |Platinum         |
|Nathan        |South     |655.92         |Platinum         |
|Victor        |South     |639.95         |Platinum         |
|Yara          |North     |635.92         |Platinum         |
|Tina          |West      |589.94         |Platinum         |
|Vince  

### Vectorized UDF (Pandas UDF)

Processes data in batches using Pandas - much faster for large datasets!

In [12]:
@pandas_udf(
    name="calculate_discount_amount",
    is_permanent=False,
    replace=True,
    input_types=[PandasSeriesType(FloatType()), PandasSeriesType(StringType())],
    return_type=PandasSeriesType(FloatType())
)
def calculate_discount_amount(spend, tier):
    """Calculate discount amount based on spend and tier."""
    import pandas as pd
    discount_rates = {"Platinum": 0.15, "Gold": 0.10, "Silver": 0.05, "Bronze": 0.00}
    rates = tier.map(discount_rates).fillna(0.0)
    return spend * rates

# Apply vectorized UDF
customer_discounts = customer_tiers.with_column(
    "DISCOUNT_AMOUNT",
    calculate_discount_amount(col("TOTAL_SPEND"), col("DISCOUNT_TIER"))
)

customer_discounts.show(10)

----------------------------------------------------------------------------------
|"FIRST_NAME"  |"REGION"  |"TOTAL_SPEND"  |"DISCOUNT_TIER"  |"DISCOUNT_AMOUNT"   |
----------------------------------------------------------------------------------
|Queenie       |East      |259.92         |Gold             |25.992000000000004  |
|Brian         |West      |1019.95        |Platinum         |152.9925            |
|Cathy         |North     |249.91         |Gold             |24.991              |
|Ursula        |East      |908.92         |Platinum         |136.338             |
|Elena         |East      |229.93         |Gold             |22.993000000000002  |
|Liam          |South     |222.93         |Gold             |22.293000000000003  |
|Felix         |West      |194.94         |Silver           |9.747               |
|Grace         |East      |181.91         |Silver           |9.0955              |
|Maya          |East      |1039.95        |Platinum         |155.9925            |
|Eve

## 7. Stored Procedures

Package the entire pipeline as a callable Stored Procedure.

In [13]:
def build_monthly_revenue_summary(session: Session) -> str:
    """Full ETL pipeline packaged as a stored procedure."""
    
    # Load
    customers = session.table("SHOPSTREAM.RAW.CUSTOMERS")
    products = session.table("SHOPSTREAM.RAW.PRODUCTS")
    orders = session.table("SHOPSTREAM.RAW.ORDERS")
    
    # Transform
    order_details = (
        orders
        .join(customers, orders["CUSTOMER_ID"] == customers["CUSTOMER_ID"])
        .join(products, orders["PRODUCT_ID"] == products["PRODUCT_ID"])
        .select(
            orders["ORDER_ID"],
            orders["ORDER_DATE"],
            customers["REGION"],
            (orders["QUANTITY"] * products["UNIT_PRICE"]).alias("ORDER_TOTAL")
        )
    )
    
    monthly_summary = (
        order_details
        .group_by(
            col("REGION"),
            date_trunc("MONTH", col("ORDER_DATE")).alias("MONTH")
        )
        .agg(
            sum("ORDER_TOTAL").alias("TOTAL_REVENUE"),
            count("ORDER_ID").alias("ORDER_COUNT"),
            avg("ORDER_TOTAL").alias("AVG_ORDER_VALUE")
        )
        .with_column(
            "REVENUE_TIER",
            when(col("TOTAL_REVENUE") > 5000, "High")
            .when(col("TOTAL_REVENUE") > 2000, "Medium")
            .otherwise("Low")
        )
    )
    
    # Save
    target = "SHOPSTREAM.ANALYTICS.MONTHLY_REVENUE_SUMMARY"
    monthly_summary.write.mode("overwrite").save_as_table(target)
    
    row_count = session.table(target).count()
    return f"SUCCESS: Saved {row_count} rows to {target}"


# Register as Stored Procedure
session.sproc.register(
    func=build_monthly_revenue_summary,
    name="build_monthly_revenue_summary_sp",
    replace=True,
    is_permanent=False
)

print("Stored Procedure registered!")

Stored Procedure registered!


In [14]:
# Call the stored procedure
result = session.sql("CALL build_monthly_revenue_summary_sp()").collect()
print(f"Result: {result[0][0]}")

# View the output
session.table("SHOPSTREAM.ANALYTICS.MONTHLY_REVENUE_SUMMARY").sort("MONTH").show(10)

Result: SUCCESS: Saved 48 rows to SHOPSTREAM.ANALYTICS.MONTHLY_REVENUE_SUMMARY
------------------------------------------------------------------------------------------------
|"REGION"  |"MONTH"     |"TOTAL_REVENUE"  |"ORDER_COUNT"  |"AVG_ORDER_VALUE"  |"REVENUE_TIER"  |
------------------------------------------------------------------------------------------------
|West      |2024-01-01  |599.96           |2              |299.98000000       |Low             |
|East      |2024-01-01  |386.94           |3              |128.98000000       |Low             |
|South     |2024-01-01  |334.95           |4              |83.73750000        |Low             |
|North     |2024-01-01  |889.91           |6              |148.31833333       |Low             |
|East      |2024-02-01  |599.88           |6              |99.98000000        |Low             |
|North     |2024-02-01  |559.94           |5              |111.98800000       |Low             |
|South     |2024-02-01  |540.96           |3    

## 8. Summary

| Concept | What You Learned |
|---------|------------------|
| **Session** | Connect to Snowflake from Python |
| **DataFrames** | Lazy evaluation, col(), filter/join/group_by/agg |
| **Scalar UDF** | Custom row-by-row Python logic |
| **Vectorized UDF** | Batch processing with Pandas (faster) |
| **Stored Procedure** | Deploy full pipelines, callable from SQL |

### Key Principles
- Everything runs on **Snowflake's compute** (not your laptop)
- DataFrames are **lazy** - nothing executes until you call `.show()` / `.collect()`
- Use the `main(session)` pattern for code that works **locally AND deployed**
- Prefer native SQL functions over UDFs when possible (performance)
- Use vectorized UDFs over scalar UDFs for large datasets

In [ ]:
# Clean up
session.close()
print("Session closed. Workshop complete!")